# VectorSpaceProximityWorkshop

## Student
- Name: Ce Chen
- StudentID: 9007166
- Group: Group1


## 1. Setup

In [1]:
import re
import math
import numpy as np
import pandas as pd

from collections import Counter
from typing import List, Dict, Iterable

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 180)
np.set_printoptions(suppress=True)

## 2. Load the Corpus

I use the **20 Newsgroups** training split as the main corpus.  
This dataset is a standard benchmark for text classification and retrieval experiments because it contains many short to medium-length documents across several topic areas.

For a retrieval assignment, it is large enough to be realistic while still manageable in a notebook.

In [2]:
newsgroups = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes")
)

corpus_df = pd.DataFrame({
    "doc_id": np.arange(len(newsgroups.data)),
    "text": newsgroups.data,
    "target": newsgroups.target,
    "category": [newsgroups.target_names[i] for i in newsgroups.target]
})

# Basic cleanup for empty or near-empty rows
corpus_df["text"] = corpus_df["text"].fillna("").astype(str)
corpus_df = corpus_df[corpus_df["text"].str.strip().str.len() > 30].reset_index(drop=True)

print("Documents in working corpus:", len(corpus_df))
print("Number of categories:", corpus_df["category"].nunique())
corpus_df.head()

Documents in working corpus: 10892
Number of categories: 20


,doc_id,text,target,category
0,0,"I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a...",7,rec.autos
1,1,A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with t...,4,comp.sys.mac.hardware
2,2,"well folks, my mac plus finally gave up the ghost this weekend after\nstarting life as a 512k way back in 1985. sooo, i'm in the market for a\nnew machine a bit sooner than i ...",4,comp.sys.mac.hardware
3,3,\nDo you have Weitek's address/phone number? I'd like to get some information\nabout this chip.\n,1,comp.graphics
4,4,"From article <C5owCB.n3p@world.std.com>, by tombaker@world.std.com (Tom A Baker):\n\n\nMy understanding is that the 'expected errors' are basically\nknown bugs in the warning s...",14,sci.space


In [3]:
# Reportable corpus statistics
token_pattern = re.compile(r"[a-zA-Z]{2,}")
approx_vocab = set()

for text in corpus_df["text"].head(3000):
    approx_vocab.update(token_pattern.findall(text.lower()))

corpus_stats = pd.DataFrame({
    "Metric": [
        "Corpus source",
        "Documents used",
        "Topic categories",
        "Approx. vocabulary size (sampled from first 3000 docs)"
    ],
    "Value": [
        "scikit-learn 20 Newsgroups",
        len(corpus_df),
        corpus_df["category"].nunique(),
        len(approx_vocab)
    ]
})

corpus_stats

,Metric,Value
0,Corpus source,scikit-learn 20 Newsgroups
1,Documents used,10892
2,Topic categories,20
3,Approx. vocabulary size (sampled from first 3000 docs),35999


### Sample Documents

In [4]:
sample_preview = corpus_df.loc[:4, ["doc_id", "category", "text"]].copy()
sample_preview["text"] = sample_preview["text"].str[:300] + "..."
sample_preview

,doc_id,category,text
0,0,rec.autos,"I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a..."
1,1,comp.sys.mac.hardware,A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with t...
2,2,comp.sys.mac.hardware,"well folks, my mac plus finally gave up the ghost this weekend after\nstarting life as a 512k way back in 1985. sooo, i'm in the market for a\nnew machine a bit sooner than i ..."
3,3,comp.graphics,\nDo you have Weitek's address/phone number? I'd like to get some information\nabout this chip.\n...
4,4,sci.space,"From article <C5owCB.n3p@world.std.com>, by tombaker@world.std.com (Tom A Baker):\n\n\nMy understanding is that the 'expected errors' are basically\nknown bugs in the warning s..."


## 3. Preprocessing Pipeline

The preprocessing pipeline includes:

- lowercasing
- punctuation and digit removal
- tokenization
- stop-word removal
- stemming

I keep the pipeline simple and explicit so each step is easy to inspect.

In [5]:
try:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    STEMMING_AVAILABLE = True
except Exception:
    stemmer = None
    STEMMING_AVAILABLE = False

stop_words = set(ENGLISH_STOP_WORDS)

def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z]{2,}", text.lower())

def normalize_tokens(tokens: List[str]) -> List[str]:
    cleaned = [t for t in tokens if t not in stop_words]
    if STEMMING_AVAILABLE:
        cleaned = [stemmer.stem(t) for t in cleaned]
    return cleaned

def preprocess(text: str) -> List[str]:
    tokens = tokenize(text)
    tokens = normalize_tokens(tokens)
    return tokens

def preprocess_to_string(text: str) -> str:
    return " ".join(preprocess(text))

print("Stemming available:", STEMMING_AVAILABLE)
print(preprocess("Machine learning methods for graphics, graphics, and image rendering in 2026!"))

Stemming available: False
['machine', 'learning', 'methods', 'graphics', 'graphics', 'image', 'rendering']


In [6]:
# Show before/after examples
preview_rows = corpus_df.loc[:2, ["doc_id", "category", "text"]].copy()
preview_rows["raw_preview"] = preview_rows["text"].str[:220] + "..."
preview_rows["processed_preview"] = preview_rows["text"].apply(lambda x: " ".join(preprocess(x)[:40]))
preview_rows[["doc_id", "category", "raw_preview", "processed_preview"]]

,doc_id,category,raw_preview,processed_preview
0,0,rec.autos,"I was wondering if anyone out there could enlighten me on this car I saw\nthe other day. It was a 2-door sports car, looked to be from the late 60s/\nearly 70s. It was called a...",wondering enlighten car saw day door sports car looked late early called bricklin doors really small addition bumper separate rest body know tellme model engine specs years pro...
1,1,comp.sys.mac.hardware,A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with t...,fair number brave souls upgraded si clock oscillator shared experiences poll send brief message detailing experiences procedure speed attained cpu rated speed add cards adapter...
2,2,comp.sys.mac.hardware,"well folks, my mac plus finally gave up the ghost this weekend after\nstarting life as a 512k way back in 1985. sooo, i'm in the market for a\nnew machine a bit sooner than i ...",folks mac plus finally gave ghost weekend starting life way sooo market new machine bit sooner intended looking picking powerbook maybe bunch questions hopefully somebody answe...


## 4. Apply Preprocessing to the Corpus

In [7]:
corpus_df["processed_text"] = corpus_df["text"].apply(preprocess_to_string)
corpus_df["token_count"] = corpus_df["processed_text"].apply(lambda s: len(s.split()))
corpus_df[["doc_id", "category", "token_count"]].head()

,doc_id,category,token_count
0,0,rec.autos,35
1,1,comp.sys.mac.hardware,48
2,2,comp.sys.mac.hardware,140
3,3,comp.graphics,7
4,4,sci.space,42


## 5. Term-Document Incidence Matrix

The incidence matrix records whether a term appears in a document at least once.  
This is the binary representation used in simple Boolean-style retrieval.

In [8]:
small_doc_sample = corpus_df["processed_text"].head(8).tolist()

binary_vectorizer = CountVectorizer(binary=True, max_features=20)
binary_matrix_small = binary_vectorizer.fit_transform(small_doc_sample)

incidence_df = pd.DataFrame(
    binary_matrix_small.toarray(),
    columns=binary_vectorizer.get_feature_names_out(),
    index=[f"Doc_{i}" for i in range(len(small_doc_sample))]
)

incidence_df

,chip,day,don,info,know,machine,mail,maybe,mb,new,news,number,people,really,right,set,speed,thanks,tom,uses
Doc_0,0,1,0,1,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0
Doc_1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0
Doc_2,0,1,1,1,1,1,0,1,1,1,1,0,1,1,0,0,0,1,1,1
Doc_3,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
Doc_4,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0
Doc_5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
Doc_6,0,0,0,1,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0
Doc_7,1,0,1,1,1,1,0,1,1,0,0,0,0,0,1,1,1,0,0,1


## 6. Term Frequency (TF)

Term frequency shows how often a term appears in one document.  
Below, I inspect raw counts and normalized counts for a sample document.

In [9]:
sample_doc = corpus_df.loc[0, "processed_text"]
sample_tokens = sample_doc.split()
tf_counts = Counter(sample_tokens)

tf_df = pd.DataFrame(tf_counts.items(), columns=["term", "raw_tf"])
tf_df["normalized_tf"] = tf_df["raw_tf"] / len(sample_tokens)
tf_df = tf_df.sort_values(["raw_tf", "term"], ascending=[False, True]).head(15).reset_index(drop=True)
tf_df

,term,raw_tf,normalized_tf
0,car,4,0.114286
1,addition,1,0.028571
2,body,1,0.028571
3,bricklin,1,0.028571
4,bumper,1,0.028571
5,called,1,0.028571
6,day,1,0.028571
7,door,1,0.028571
8,doors,1,0.028571
9,early,1,0.028571


## 7. Log TF

Log weighting reduces the gap between medium-frequency and very high-frequency terms.

In [10]:
tf_df["log_tf"] = tf_df["raw_tf"].apply(lambda x: 1 + math.log10(x) if x > 0 else 0)
tf_df

,term,raw_tf,normalized_tf,log_tf
0,car,4,0.114286,1.60206
1,addition,1,0.028571,1.00000
2,body,1,0.028571,1.00000
3,bricklin,1,0.028571,1.00000
4,bumper,1,0.028571,1.00000
5,called,1,0.028571,1.00000
6,day,1,0.028571,1.00000
7,door,1,0.028571,1.00000
8,doors,1,0.028571,1.00000
9,early,1,0.028571,1.00000


## 8. Document Frequency (DF) and Inverse Document Frequency (IDF)

DF counts in how many documents a term appears.  
IDF gives more weight to terms that occur in fewer documents.

In [11]:
count_vectorizer_df = CountVectorizer(max_features=3000)
count_matrix_df = count_vectorizer_df.fit_transform(corpus_df["processed_text"])

terms_df = count_vectorizer_df.get_feature_names_out()
doc_freq = np.asarray((count_matrix_df > 0).sum(axis=0)).ravel()
N = count_matrix_df.shape[0]
idf_values = np.log10(N / doc_freq)

df_idf_table = pd.DataFrame({
    "term": terms_df,
    "df": doc_freq,
    "idf": idf_values
}).sort_values("idf", ascending=False)

# Show both rare and common terms
rare_terms = df_idf_table.head(10).copy()
common_terms = df_idf_table.sort_values("idf", ascending=True).head(10).copy()

print("Rare terms with high IDF")
display(rare_terms)

print("Common terms with low IDF")
display(common_terms)

Rare terms with high IDF


,term,df,idf
1831,oname,2,3.736078
2350,scx,2,3.736078
1069,gcx,3,3.559986
1028,fprintf,3,3.559986
1702,mov,3,3.559986
2158,rck,4,3.435048
2281,rmc,4,3.435048
59,ahf,4,3.435048
2831,uww,4,3.435048
95,anonymity,4,3.435048


Common terms with low IDF


,term,df,idf
1488,like,2549,0.630738
1380,just,2456,0.646879
760,don,2326,0.670498
1414,know,2284,0.678412
2691,think,1893,0.759957
754,does,1824,0.776083
1920,people,1818,0.777514
2707,time,1745,0.795312
2812,use,1649,0.819887
1105,good,1578,0.839001


## 9. TF-IDF Construction

For large-scale retrieval, I use `TfidfVectorizer` after the same preprocessing logic.  
This keeps the representation efficient and consistent with the earlier manual steps.

In [12]:
def identity_tokenizer(text):
    return text.split()

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=5000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_df["processed_text"])
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of features:", len(tfidf_terms))

TF-IDF matrix shape: (10892, 5000)
Number of features: 5000


In [13]:
# Show top TF-IDF weights for one sample document
doc_index = 0
doc_vector = tfidf_matrix[doc_index].toarray().ravel()
top_idx = np.argsort(doc_vector)[::-1][:15]

top_tfidf_terms = pd.DataFrame({
    "term": tfidf_terms[top_idx],
    "tfidf_weight": doc_vector[top_idx]
})

top_tfidf_terms

,term,tfidf_weight
0,car,0.574999
1,doors,0.213621
2,sports,0.202690
3,specs,0.202203
4,production,0.199869
5,door,0.183744
6,separate,0.180818
7,engine,0.177202
8,addition,0.174147
9,late,0.170968


## 10. Retrieval Functions

I compare two document representations:

1. **Binary incidence**
2. **TF-IDF**

Both use cosine similarity for ranking.

In [14]:
binary_retrieval_vectorizer = CountVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    binary=True,
    max_features=5000
)

binary_retrieval_matrix = binary_retrieval_vectorizer.fit_transform(corpus_df["processed_text"])

def preprocess_query(query: str) -> str:
    return preprocess_to_string(query)

def retrieve_binary(query: str, top_k: int = 5) -> pd.DataFrame:
    q = preprocess_query(query)
    q_vec = binary_retrieval_vectorizer.transform([q])
    scores = cosine_similarity(q_vec, binary_retrieval_matrix).ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = corpus_df.loc[top_idx, ["doc_id", "category", "text"]].copy()
    results["score"] = scores[top_idx]
    results["representation"] = "Binary"
    results["text_preview"] = results["text"].str[:240].str.replace("\n", " ", regex=True) + "..."
    return results[["representation", "doc_id", "category", "score", "text_preview"]].reset_index(drop=True)

def retrieve_tfidf(query: str, top_k: int = 5) -> pd.DataFrame:
    q = preprocess_query(query)
    q_vec = tfidf_vectorizer.transform([q])
    scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = corpus_df.loc[top_idx, ["doc_id", "category", "text"]].copy()
    results["score"] = scores[top_idx]
    results["representation"] = "TF-IDF"
    results["text_preview"] = results["text"].str[:240].str.replace("\n", " ", regex=True) + "..."
    return results[["representation", "doc_id", "category", "score", "text_preview"]].reset_index(drop=True)

## 11. Information Needs and Query Set

I define five information needs and match each one with a short query.  
The queries are intentionally simple so that the ranking behavior is easy to inspect.

In [15]:
query_bank = [
    {
        "query_id": "Q1",
        "information_need": "Find discussions about space missions, orbit, or NASA-related topics.",
        "query": "nasa space shuttle orbit mission",
        "relevant_categories": ["sci.space"]
    },
    {
        "query_id": "Q2",
        "information_need": "Find discussions about computer graphics and image rendering.",
        "query": "computer graphics image rendering 3d",
        "relevant_categories": ["comp.graphics"]
    },
    {
        "query_id": "Q3",
        "information_need": "Find discussions about baseball teams, games, or league performance.",
        "query": "baseball game team season pitching",
        "relevant_categories": ["rec.sport.baseball"]
    },
    {
        "query_id": "Q4",
        "information_need": "Find discussions about hockey players, games, or NHL-style topics.",
        "query": "hockey game team season playoffs",
        "relevant_categories": ["rec.sport.hockey"]
    },
    {
        "query_id": "Q5",
        "information_need": "Find discussions about religion and atheism debates.",
        "query": "atheism religion belief god debate",
        "relevant_categories": ["alt.atheism", "talk.religion.misc", "soc.religion.christian"]
    }
]

query_df = pd.DataFrame(query_bank)
query_df

,query_id,information_need,query,relevant_categories
0,Q1,"Find discussions about space missions, orbit, or NASA-related topics.",nasa space shuttle orbit mission,[sci.space]
1,Q2,Find discussions about computer graphics and image rendering.,computer graphics image rendering 3d,[comp.graphics]
2,Q3,"Find discussions about baseball teams, games, or league performance.",baseball game team season pitching,[rec.sport.baseball]
3,Q4,"Find discussions about hockey players, games, or NHL-style topics.",hockey game team season playoffs,[rec.sport.hockey]
4,Q5,Find discussions about religion and atheism debates.,atheism religion belief god debate,"[alt.atheism, talk.religion.misc, soc.religion.christian]"


## 12. Retrieval Demo

For each query, I show the top ranked results under both representations.

In [16]:
for item in query_bank:
    print("=" * 110)
    print(item["query_id"], "-", item["information_need"])
    print("Query:", item["query"])
    print("\nTop 5 with Binary representation")
    display(retrieve_binary(item["query"], top_k=5))
    print("\nTop 5 with TF-IDF representation")
    display(retrieve_tfidf(item["query"], top_k=5))

Q1 - Find discussions about space missions, orbit, or NASA-related topics.
Query: nasa space shuttle orbit mission

Top 5 with Binary representation


,representation,doc_id,category,score,text_preview
0,Binary,6502,comp.graphics,0.316228,Look in the /pub/SPACE directory on ames.arc.nasa.gov - there are a number of earth images there. You may have to hunt around the subdirectories as things tend to be filed unde...
1,Binary,3044,sci.space,0.311400,Better idea for use of NASA Shuttle Astronauts and Crew is have them be found lost in space after a accident with a worm hole or other space/time glitch.. Maybe age Jemison a...
2,Binary,4443,sci.space,0.286039,"I am coordinating the Space Shuttle Program Office's e-mail traffic to NPO Energia for our on-going Joint Missions. I have several e-mail addresses for NPO Energia folks, but ..."
3,Binary,4312,sci.space,0.269680,There is a guy in NASA Johnson Space Center that might answer your question. I do not have his name right now but if you follow up I can dig that out for you. C.O.Egalon@la...
4,Binary,6148,sci.space,0.263752,"I'm on a fact-finding mission, trying to find out if there exists a list of potentially world-bearing stars within 100 light years of the Sun... Is anyone currently ..."



Top 5 with TF-IDF representation


,representation,doc_id,category,score,text_preview
0,TF-IDF,153,sci.space,0.502441,"Archive-name: space/schedule Last-modified: $Date: 93/04/01 14:39:23 $ SPACE SHUTTLE ANSWERS, LAUNCH SCHEDULES, TV COVERAGE SHUTTLE LAUNCHINGS AND LANDINGS; SCHEDULES AND..."
1,TF-IDF,11198,sci.space,0.492116,Archive-name: space/controversy Last-modified: $Date: 93/04/01 14:39:06 $ CONTROVERSIAL QUESTIONS These issues periodically come up with much argument and few facts being...
2,TF-IDF,5880,sci.space,0.464623,"Ed Campion Headquarters, Washington, D.C. April 23, 1993 (Phone: 202/358-1780) Kyle Herring Johnson Space Center, Houston (Phone: 713/483-5111) ..."
3,TF-IDF,3044,sci.space,0.446189,Better idea for use of NASA Shuttle Astronauts and Crew is have them be found lost in space after a accident with a worm hole or other space/time glitch.. Maybe age Jemison a...
4,TF-IDF,4443,sci.space,0.432325,"I am coordinating the Space Shuttle Program Office's e-mail traffic to NPO Energia for our on-going Joint Missions. I have several e-mail addresses for NPO Energia folks, but ..."


Q2 - Find discussions about computer graphics and image rendering.
Query: computer graphics image rendering 3d

Top 5 with Binary representation


...

,representation,doc_id,category,score,text_preview
0,TF-IDF,7995,rec.sport.hockey,0.476155,"But only in NY,NJ, Philadelphia, and Chicago. Everywhere else, the only reason SportsChannel was available was for local baseball broadcasts. And local baseball pre-empted the..."
1,TF-IDF,494,rec.sport.hockey,0.461411,"Nice try Deepak, but ""tough Whaler squad"" should have clued you in to the fact that my Leaf woofing was tongue-in-cheek. If playoff hockey is any more intense than ..."
2,TF-IDF,7307,rec.sport.hockey,0.446515,[more about the Messier-Samuelsson incident] I agree with Rick that Ulf's cross check wasn't illegal. It was the kind of check you see a dozen times during a game without bei...
3,TF-IDF,9319,rec.sport.hockey,0.428277,And last year the Capitals had the Pens number up until about game 3 of the playoffs. ...
4,TF-IDF,6985,rec.sport.hockey,0.381470,"As the Sharks' season came to a close tonight, I will start a series of posts, trying to revisit the players, the trades, the moves, etc., that went through for the Sharks for ..."


Q5 - Find discussions about religion and atheism debates.
Query: atheism religion belief god debate

Top 5 with Binary representation


,representation,doc_id,category,score,text_preview
0,Binary,6492,alt.atheism,0.365148,Since when does atheism mean trashing other religions?There must be a God of inbreeding to which you are his only son....
1,Binary,1228,soc.religion.christian,0.316228,"He who overcomes will inherit all this, and I will be his God and he will be my son. ..."
2,Binary,10878,alt.atheism,0.279751,"For the last time, Bobby. Lack of belief in YOUR god does NOT imply atheism. Just because some moslems aren't moral does not mean they don't believe in a god named Allah, alt..."
3,Binary,7998,alt.atheism,0.258199,^^^^^^^^^^^^^^^ a) I think that he has a rather witty .sig file. It sums up a great deal of atheistic thought (IMO) in one sim...
4,Binary,6746,talk.religion.misc,0.258199,"\tUnless God admits that he didn't do it.... \t=) --- "" I'd Cheat on Hillary Too.""..."



Top 5 with TF-IDF representation


,representation,doc_id,category,score,text_preview
0,TF-IDF,3633,alt.atheism,0.622137,"<In article <31MAR199321091163@juliet.caltech.edu<, lmh@juliet.caltech.edu (Henling, Lawrence M.) writes... <<Atheism (Greek 'a' not + 'theos' god) Belief that there is no god..."
1,TF-IDF,10924,alt.atheism,0.514611,":P>My atheism is incidental, and the question of ""God"" is trivial. :P :P>But........ :P :P>It matters a great deal to me when idiots try to force their belief on me, :P>when th..."
2,TF-IDF,7539,alt.atheism,0.454367,"Yes I fully agree with that, but is it ""I don't believe gods exist"", or ""I believe no gods exist""? As MANDTBACKA@FINABO.ABO.FI (Mats Andtbacka) pointed out, it all hinges on ..."
3,TF-IDF,5200,alt.atheism,0.433953,Archive-name: atheism/introduction Alt-atheism-archive-name: introduction Last-modified: 5 April 1993 Version: 1.2 -----BEGIN PGP SIGNED MESSAGE----- ...
4,TF-IDF,10194,alt.atheism,0.374100,[deleted] think: [deleted] ^^^^^^^^^^^^^^^^^^^^^^^^ ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ ^^^^^^^^^^^^...


## 13. Direct Representation Comparison

This section makes the difference between binary matching and TF-IDF ranking easier to see for one query.

In [17]:
comparison_query = query_bank[0]["query"]  # Q1
binary_results = retrieve_binary(comparison_query, top_k=10)
tfidf_results = retrieve_tfidf(comparison_query, top_k=10)

comparison_table = pd.concat([binary_results, tfidf_results], ignore_index=True)
comparison_table

,representation,doc_id,category,score,text_preview
0,Binary,6502,comp.graphics,0.316228,Look in the /pub/SPACE directory on ames.arc.nasa.gov - there are a number of earth images there. You may have to hunt around the subdirectories as things tend to be filed unde...
1,Binary,3044,sci.space,0.311400,Better idea for use of NASA Shuttle Astronauts and Crew is have them be found lost in space after a accident with a worm hole or other space/time glitch.. Maybe age Jemison a...
2,Binary,4443,sci.space,0.286039,"I am coordinating the Space Shuttle Program Office's e-mail traffic to NPO Energia for our on-going Joint Missions. I have several e-mail addresses for NPO Energia folks, but ..."
3,Binary,4312,sci.space,0.269680,There is a guy in NASA Johnson Space Center that might answer your question. I do not have his name right now but if you follow up I can dig that out for you. C.O.Egalon@la...
4,Binary,6148,sci.space,0.263752,"I'm on a fact-finding mission, trying to find out if there exists a list of potentially world-bearing stars within 100 light years of the Sun... Is anyone currently ..."
5,Binary,3534,sci.space,0.258199,"Even worse, the city of Atlanta has a proposal before it to rent space on this orbiting billboard. Considering the caliber of people running this city, there's no telling w..."
6,Binary,10219,sci.space,0.243432,": >There is an emergency oxygen system that is capable of maintaining a : >breathable atmosphere in the cabin for long enough to come down, even : >if there is something like ..."
7,Binary,2624,sci.space,0.239046,"I am looking for any information about the space program. This includes NASA, the shuttles, history, anything! I would like to know if anyone could suggest books, periodicals,..."
8,Binary,6869,sci.space,0.233550,": Has anyone ever heard of a food product called ""Space Food Sticks?"" I remember those awful things. They were dry and crumbly, and I recall asking my third-grade teacher, M..."
9,Binary,11107,sci.space,0.230940,"I gues it is Keesler. The others do not ring the bell but they might be involved as well. Sometime ago Keesler was here at Langley teaching a course on space debris and, if ..."


### Short Observation

Binary matching only checks whether query terms appear, so many results can receive similar scores.  
TF-IDF usually separates the results better because it rewards more informative terms and downweights broadly shared ones.

## 14. Evaluation Design

The workshop asks for evaluation on at least three queries.  
To keep the judgments consistent and reproducible, I use **category-based relevance labels**:

- a retrieved document is treated as relevant if its newsgroup category belongs to the query's target category set
- otherwise it is treated as nonrelevant

This is not the same as a fully manual pooling process, but it gives a clear and defensible benchmark for this assignment.

In [18]:
eval_queries = query_bank[:3]  # Evaluate Q1, Q2, and Q3

def build_ranked_results(query: str, representation: str, top_k: int = 10) -> pd.DataFrame:
    if representation == "Binary":
        q = preprocess_query(query)
        q_vec = binary_retrieval_vectorizer.transform([q])
        scores = cosine_similarity(q_vec, binary_retrieval_matrix).ravel()
    elif representation == "TF-IDF":
        q = preprocess_query(query)
        q_vec = tfidf_vectorizer.transform([q])
        scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    else:
        raise ValueError("representation must be 'Binary' or 'TF-IDF'")

    top_idx = np.argsort(scores)[::-1][:top_k]
    ranked = corpus_df.loc[top_idx, ["doc_id", "category", "text"]].copy()
    ranked["score"] = scores[top_idx]
    ranked["rank"] = np.arange(1, len(ranked) + 1)
    ranked["text_preview"] = ranked["text"].str[:180].str.replace("\n", " ", regex=True) + "..."
    return ranked.reset_index(drop=True)

def attach_relevance(ranked_df: pd.DataFrame, relevant_categories: List[str]) -> pd.DataFrame:
    judged = ranked_df.copy()
    judged["relevant"] = judged["category"].isin(relevant_categories).astype(int)
    return judged

In [19]:
# Show judged ranked lists for the three evaluation queries
for item in eval_queries:
    print("=" * 110)
    print(item["query_id"], "-", item["query"])
    judged = attach_relevance(
        build_ranked_results(item["query"], "TF-IDF", top_k=10),
        item["relevant_categories"]
    )
    display(judged[["rank", "doc_id", "category", "score", "relevant", "text_preview"]])

Q1 - nasa space shuttle orbit mission


,rank,doc_id,category,score,relevant,text_preview
0,1,153,sci.space,0.502441,1,"Archive-name: space/schedule Last-modified: $Date: 93/04/01 14:39:23 $ SPACE SHUTTLE ANSWERS, LAUNCH SCHEDULES, TV COVERAGE SHUTTLE LAUNCHINGS AND LANDINGS; SCHEDULES AND..."
1,2,11198,sci.space,0.492116,1,Archive-name: space/controversy Last-modified: $Date: 93/04/01 14:39:06 $ CONTROVERSIAL QUESTIONS These issues periodically come up with much argument and few facts being...
2,3,5880,sci.space,0.464623,1,"Ed Campion Headquarters, Washington, D.C. April 23, 1993 (Phone: 202/358-1780) Kyle Herring Johnson Space Center, Houston (Phone: 713/483-5111) ..."
3,4,3044,sci.space,0.446189,1,Better idea for use of NASA Shuttle Astronauts and Crew is have them be found lost in space after a accident with a worm hole or other space/time glitch.. Maybe age Jemison a...
4,5,4443,sci.space,0.432325,1,"I am coordinating the Space Shuttle Program Office's e-mail traffic to NPO Energia for our on-going Joint Missions. I have several e-mail addresses for NPO Energia folks, but ..."
5,6,4425,sci.space,0.431619,1,"Archive-name: space/addresses Last-modified: $Date: 93/04/01 14:38:55 $ CONTACTING NASA, ESA, AND OTHER SPACE AGENCIES/COMPANIES Many space activities center around large Gov..."
6,7,9096,sci.space,0.416417,1,Archive-name: space/intro Last-modified: $Date: 93/04/01 14:39:10 $ FREQUENTLY ASKED QUESTIONS ON SCI.SPACE/SCI.ASTRO INTRODUCTION This series of linked messages...
7,8,2800,sci.space,0.387151,1,Archive-name: space/net Last-modified: $Date: 93/04/01 14:39:15 $ NETWORK RESOURCES OVERVIEW You may be reading this document on any one of an amazing variety of com...
8,9,545,sci.space,0.361194,1,"Archive-name: space/astronaut Last-modified: $Date: 93/04/01 14:39:02 $ HOW TO BECOME AN ASTRONAUT First the short form, authored by Henry Spencer, then an official NASA ..."
9,10,10165,sci.space,0.357725,1,I have 19 (2 MB worth!) uuencode'd GIF images contain charts outlining one of the many alternative Space Station designs being considered in Crystal City. Mr. Mark Holderman w...


Q2 - computer graphics image rendering 3d


,rank,doc_id,category,score,relevant,text_preview
0,1,5432,comp.graphics,0.389202,1,"My package is based on several articles about non-standard radiosity and some unpublished methods. The main articles are: - Cohen, Chen, Wallace, Greenberg : A Progres..."
1,2,5704,comp.graphics,0.362810,1,I recently got a file describing a library of rendering routines called SIPP (SImple Polygon Processor). Could anyone tell me where I can FTP the source code and which is th...
2,3,3757,comp.graphics,0.341791,1,"Graeme> \tYes, that's known as ""Bresenhams Run Length Slice Algorithm for Graeme> Incremental lines"". See Fundamental Algorithms for Computer Graphics, Graeme> Springer-Verlag..."
3,4,2915,comp.graphics,0.335480,1,"\tYes, that's known as ""Bresenhams Run Length Slice Algorithm for Incremental lines"". See Fundamental Algorithms for Computer Graphics, Springer-Verlag, Berlin Heidelberg 1985..."
4,5,639,comp.graphics,0.325316,1,"Stone, DeRose: Geometric characterization of parametric cubic curves. ACM Trans. Graphics 8 (3) (1989) 147 - 163. Manocha, Canny: Detecting cusps and inflection points in c..."
5,6,4166,comp.graphics,0.324314,1,Archive-name: graphics/resources-list/part2 Last-modified: 1993/04/17 Computer Graphics Resource Listing : WEEKLY POSTING [ PART 2/3 ] =======================================...
6,7,4589,comp.graphics,0.318068,1,"Sorry I missed you Raymond, I was just out in Dahlgren last month... I'm the Virtual Reality market manager for Silicon Graphics, so perhaps I can help a little. Unfortunat..."
7,8,6197,comp.graphics,0.298076,1,"**************************** SPHINX *************************** \tSphinx is a user-friendly, state-of-the-art image processing and analysis package that runs across a spec..."
8,9,8148,comp.graphics,0.283629,1,"Hello, I am searching for rendering software which has been developed to specifically take advantage of multi-processor computer systems. Any pointers to such software wou..."
9,10,4583,comp.graphics,0.281456,1,"Check out Image Pals v1.2 from U-Lead (until May, special $99 intro price, 310-523-9393). It has the basic image processing tools for all major formats, does screen grabbing, a..."


Q3 - baseball game team season pitching


,rank,doc_id,category,score,relevant,text_preview
0,1,1606,rec.sport.baseball,0.453022,1,"It's not a cliche, and (unlike your comments below) it's not a tautology. It needn't have been true. If every pitcher in baseball were essentially the same in quality (i.e...."
1,2,1485,rec.sport.baseball,0.435759,1,"This was (my opinion) the stupidest thing in the Hidden Game. The argument was 1) Defense, or runs allowed, is 50% of the game. 2) Unearned runs amount to 12% of the runs all..."
2,3,9268,rec.sport.baseball,0.424906,1,But you still need the pitching staff to hold the opposing team to one run....
3,4,4214,rec.sport.baseball,0.417126,1,"Not if you've scored four runs, you don't! Why strain even the best pitching staff? Why not make it easier for them? In the 2-1 game, the best pitching staff in the wor..."
4,5,9074,rec.sport.baseball,0.414824,1,"The Orioles' pitching staff again is having a fine exhibition season. Four shutouts, low team ERA, (Well, I haven't gotten any baseball news since March 14 but anyways) Could t..."
5,6,9176,rec.sport.baseball,0.410614,1,"Would you say the same thing about the Dodgers in '65 or '66? True, Cone is probably as good as Drysdale, and they have no Koufax, but still, these teams were winning w..."
6,7,9392,rec.sport.baseball,0.405009,1,Has David Wells landed with a team yet? I'd think the Tigers with their anemic pitching would grab this guy pronto!...
7,8,7307,rec.sport.hockey,0.393065,0,[more about the Messier-Samuelsson incident] I agree with Rick that Ulf's cross check wasn't illegal. It was the kind of check you see a dozen times during a game without bei...
8,9,6032,rec.sport.baseball,0.374113,1,"The O's just lost to the Rangers a few minutes ago I was not too happy about the pitching of Rick Sutcliffe (6 runs in 6 innings, 5 in the 3?) This puts me in remembering the 1..."
9,10,2645,rec.sport.baseball,0.364947,1,"These people were very silly. Any team that gets to the World Series can win the World Series, and anybody who ever expects a sweep is crazy. If you put the best team in..."


## 15. Metric Functions

In [20]:
def confusion_counts(relevance_list: List[int], total_relevant_in_corpus: int) -> Dict[str, int]:
    relevance_array = np.array(relevance_list)
    tp = int(relevance_array.sum())
    fp = int((relevance_array == 0).sum())
    fn = int(total_relevant_in_corpus - tp)
    tn = int(len(corpus_df) - total_relevant_in_corpus - fp)
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def precision_at_k(relevance_list: List[int], k: int) -> float:
    rel = np.array(relevance_list[:k])
    return float(rel.sum() / k) if k > 0 else 0.0

def precision_score(relevance_list: List[int]) -> float:
    rel = np.array(relevance_list)
    return float(rel.sum() / len(rel)) if len(rel) else 0.0

def recall_score(relevance_list: List[int], total_relevant_in_corpus: int) -> float:
    rel = np.array(relevance_list)
    return float(rel.sum() / total_relevant_in_corpus) if total_relevant_in_corpus > 0 else 0.0

def f1_score_custom(precision: float, recall: float) -> float:
    return (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

def average_precision(relevance_list: List[int]) -> float:
    rel = np.array(relevance_list)
    relevant_positions = np.where(rel == 1)[0]
    if len(relevant_positions) == 0:
        return 0.0
    precisions = []
    for idx in relevant_positions:
        precisions.append(rel[:idx + 1].sum() / (idx + 1))
    return float(np.mean(precisions))

def reciprocal_rank(relevance_list: List[int]) -> float:
    for i, rel in enumerate(relevance_list, start=1):
        if rel == 1:
            return 1 / i
    return 0.0

## 16. Evaluate Binary vs TF-IDF on Three Queries

In [21]:
evaluation_rows = []

for item in eval_queries:
    total_relevant = int(corpus_df["category"].isin(item["relevant_categories"]).sum())

    for representation in ["Binary", "TF-IDF"]:
        ranked = build_ranked_results(item["query"], representation, top_k=10)
        judged = attach_relevance(ranked, item["relevant_categories"])
        rel_list = judged["relevant"].tolist()

        counts = confusion_counts(rel_list, total_relevant)
        p = precision_score(rel_list)
        r = recall_score(rel_list, total_relevant)
        f1 = f1_score_custom(p, r)
        p5 = precision_at_k(rel_list, 5)
        p10 = precision_at_k(rel_list, 10)
        ap = average_precision(rel_list)
        rr = reciprocal_rank(rel_list)

        evaluation_rows.append({
            "query_id": item["query_id"],
            "representation": representation,
            "TP": counts["TP"],
            "FP": counts["FP"],
            "FN": counts["FN"],
            "TN": counts["TN"],
            "Precision": p,
            "Recall": r,
            "F1": f1,
            "P@5": p5,
            "P@10": p10,
            "AP": ap,
            "RR": rr
        })

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df.round(4)

,query_id,representation,TP,FP,FN,TN,Precision,Recall,F1,P@5,P@10,AP,RR
0,Q1,Binary,9,1,563,10319,0.9,0.0157,0.0309,0.8,0.9,0.7857,0.5
1,Q1,TF-IDF,10,0,562,10320,1.0,0.0175,0.0344,1.0,1.0,1.0000,1.0
2,Q2,Binary,5,5,560,10322,0.5,0.0088,0.0174,0.4,0.5,0.6476,1.0
3,Q2,TF-IDF,10,0,555,10327,1.0,0.0177,0.0348,1.0,1.0,1.0000,1.0
4,Q3,Binary,5,5,554,10328,0.5,0.0089,0.0176,0.8,0.5,0.9429,1.0
5,Q3,TF-IDF,9,1,550,10332,0.9,0.0161,0.0316,1.0,0.9,0.9765,1.0


## 17. Mean Reciprocal Rank (MRR)

In [22]:
mrr_df = (
    evaluation_df
    .groupby("representation", as_index=False)["RR"]
    .mean()
    .rename(columns={"RR": "MRR"})
)

mrr_df.round(4)

,representation,MRR
0,Binary,0.8333
1,TF-IDF,1.0000


## 18. Confusion Matrix Example

Below is one confusion-matrix style summary for **Q1 with TF-IDF**.

In [23]:
example_row = evaluation_df[
    (evaluation_df["query_id"] == "Q1") &
    (evaluation_df["representation"] == "TF-IDF")
].iloc[0]

confusion_table = pd.DataFrame(
    [
        [example_row["TP"], example_row["FN"], example_row["TP"] + example_row["FN"]],
        [example_row["FP"], example_row["TN"], example_row["FP"] + example_row["TN"]],
        [example_row["TP"] + example_row["FP"], example_row["FN"] + example_row["TN"], len(corpus_df)]
    ],
    index=["Relevant", "Nonrelevant", "Total"],
    columns=["Retrieved", "Not Retrieved", "Total"]
)

confusion_table

,Retrieved,Not Retrieved,Total
Relevant,10,562,572
Nonrelevant,0,10320,10320
Total,10,10882,10892


## 19. Summary of Results

A quick average across the three evaluation queries makes the comparison easier to read.

In [24]:
summary_table = (
    evaluation_df
    .groupby("representation", as_index=False)[["Precision", "Recall", "F1", "P@5", "P@10", "AP", "RR"]]
    .mean()
    .sort_values("AP", ascending=False)
)

summary_table.round(4)

,representation,Precision,Recall,F1,P@5,P@10,AP,RR
1,TF-IDF,0.9667,0.0171,0.0336,1.0000,0.9667,0.9922,1.0000
0,Binary,0.6333,0.0112,0.0220,0.6667,0.6333,0.7920,0.8333


Save Output

In [26]:
# Save results to CSV files
evaluation_df.round(4).to_csv("vector_space_evaluation_results.csv", index=False)
summary_table.round(4).to_csv("vector_space_summary_results.csv", index=False)

print("Saved:")
print("- vector_space_evaluation_results.csv")
print("- vector_space_summary_results.csv")

Saved:
- vector_space_evaluation_results.csv
- vector_space_summary_results.csv


## 20. Reflection

### 1. Which representation worked best and why?
TF-IDF worked better in this notebook. It usually produced cleaner rankings because it gave less influence to very common terms and more influence to topic-specific words.

### 2. Did TF-IDF improve over raw or binary term counts?
Yes. Binary matching was useful as a baseline, but it often treated many documents as similarly relevant. TF-IDF separated the stronger matches more clearly.

### 3. What kinds of false positives appeared?
Some false positives came from documents that reused overlapping vocabulary without matching the actual topic focus. For example, a document might contain terms like *space* or *game* in a broad or incidental way.

### 4. What relevant documents were missed?
Relevant documents were missed when the query used only a narrow slice of the category vocabulary. A document could still be about the right topic but use different terms than the query.

### 5. How did the evaluation metrics help?
Precision, P@5, and AP were the most useful for this task because they reflected ranking quality near the top of the list. Recall was still informative, but in a top-k retrieval setting it changed more slowly because the relevant set in the corpus was much larger than the retrieved set.

### 6. How could the system be improved?
The next improvement would be to test lemmatization, tune the stop-word list, and compare TF-IDF with learned embeddings. A stronger evaluation design would also include manual relevance judgments from pooled results rather than category-based proxy labels alone.